# 01 - Dataset Preparation and Balancing

This notebook prepares and formats all datasets required for the experiments.

Outputs:
- compact JSONL datasets stored in:
  `artifacts/prepared_data/`
- dataset description file:
  `data_manifest.json`

The HDFS dataset is balanced by the two target classes:
- `OK`
- `Anomaly`

Dataset sizes and sampling parameters can be modified in the first configuration cell.

In [ ]:
import exp_common as E
from pathlib import Path
import json, pandas as pd
import random
from datasets import load_dataset

LOGS_TRAIN_PER_CLASS = 20000   
LOGS_TEST_PER_CLASS = 2500     
QA_TRAIN_N = 8000             
QA_TEST_N = 500          
CPT_N = 10000
VAL_RATIO = 0.025
PREP_SEED = 42

In [ ]:
required = [
    E.DATA_DIR/'hdfs_train.jsonl', E.DATA_DIR/'hdfs_test.jsonl',
    E.DATA_DIR/'cpt_corpus.jsonl'
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing files: ' + ', '.join(missing))
print('All source files found')

In [ ]:
def split_train_val(data, val_ratio, seed):
    rng = random.Random(seed)
    shuffled = data.copy()
    rng.shuffle(shuffled)
    n_val = int(len(shuffled) * val_ratio)
    return shuffled[n_val:], shuffled[:n_val]  # train, val

In [ ]:
print("\n=== Loading ArXiv QA Dataset ===")
try:
    arxiv_dataset = load_dataset("Navyasri12355/arxiv-qa-dataset", split="train")
    print(f"Total examples in ArXiv: {len(arxiv_dataset)}")
except Exception as e:
    print(f"Error loading ArXiv dataset: {e}")
    print("Falling back to stackexchange dataset...")
    arxiv_dataset = None
    qa_full = E.sample_rows(E.DATA_DIR/'stackexchange_train.jsonl', QA_TRAIN_N, PREP_SEED)
    qa_test = E.sample_rows(E.DATA_DIR/'stackexchange_test.jsonl', QA_TEST_N, PREP_SEED)

def convert_arxiv_to_format(example):
    if 'input' in example and 'output' in example:
        return {'input': example['input'], 'output': example['output']}
    elif 'question' in example and 'answer' in example:
        return {
            'input': f"Title: {example.get('title', '')}\n\nQuestion:\n{example['question']}",
            'output': example['answer']
        }
    else:
        # Fallback
        input_text = example.get('input') or example.get('question') or example.get('text') or str(example)
        output_text = example.get('output') or example.get('answer') or example.get('response') or ""
        return {'input': input_text, 'output': output_text}

if arxiv_dataset is not None:
    arxiv_data = [convert_arxiv_to_format(ex) for ex in arxiv_dataset]
    
    rng = random.Random(PREP_SEED)
    rng.shuffle(arxiv_data)
    
    total = len(arxiv_data)
    test_size = QA_TEST_N
    val_size = int(500)
    train_size = min(QA_TRAIN_N, total - test_size - val_size)
    
    qa_test = arxiv_data[:test_size]
    qa_val = arxiv_data[test_size:test_size + val_size]
    qa_train = arxiv_data[test_size + val_size:test_size + val_size + train_size]
    
    print(f"ArXiv QA split:")
    print(f"  Train: {len(qa_train)}")
    print(f"  Val: {len(qa_val)}")
    print(f"  Test: {len(qa_test)}")
else:
    qa_full = E.sample_rows(E.DATA_DIR/'stackexchange_train.jsonl', QA_TRAIN_N, PREP_SEED)
    qa_train, qa_val = split_train_val(qa_full, VAL_RATIO, PREP_SEED)
    qa_test = E.sample_rows(E.DATA_DIR/'stackexchange_test.jsonl', QA_TEST_N, PREP_SEED)
    print(f"StackExchange QA split:")
    print(f"  Train: {len(qa_train)}")
    print(f"  Val: {len(qa_val)}")
    print(f"  Test: {len(qa_test)}")

In [ ]:
print("\n=== Loading Logs ===")

logs_full = E.sample_balanced_logs(E.DATA_DIR/'hdfs_train.jsonl', LOGS_TRAIN_PER_CLASS, PREP_SEED)
logs_train, logs_val = split_train_val(logs_full, VAL_RATIO, PREP_SEED)
logs_test = E.sample_balanced_logs(E.DATA_DIR/'hdfs_test.jsonl', LOGS_TEST_PER_CLASS, PREP_SEED)

print(f"Logs: train={len(logs_train)}, val={len(logs_val)}, test={len(logs_test)}")

In [ ]:
print("\n=== Loading CPT ===")
cpt_full = E.sample_rows(E.DATA_DIR/'cpt_corpus.jsonl', CPT_N, PREP_SEED)
cpt_train, cpt_val = split_train_val(cpt_full, VAL_RATIO, PREP_SEED)
print(f"CPT: train={len(cpt_train)}, val={len(cpt_val)}")

In [ ]:
print("\n=== Saving prepared data ===")

E.write_jsonl([E.format_sft_example(r, 'logs') for r in logs_train], E.PREPARED_DIR/'logs_train_sft.jsonl')
E.write_jsonl([E.format_sft_example(r, 'logs') for r in logs_val],   E.PREPARED_DIR/'logs_val_sft.jsonl')
E.write_jsonl(logs_test, E.PREPARED_DIR/'logs_test_eval.jsonl')

E.write_jsonl([E.format_sft_example(r, 'qa') for r in qa_train], E.PREPARED_DIR/'qa_train_sft.jsonl')
E.write_jsonl([E.format_sft_example(r, 'qa') for r in qa_val],   E.PREPARED_DIR/'qa_val_sft.jsonl')
E.write_jsonl(qa_test, E.PREPARED_DIR/'qa_test_eval.jsonl')

E.write_jsonl([E.format_cpt_example(r) for r in cpt_train], E.PREPARED_DIR/'cpt_train.jsonl')
E.write_jsonl([E.format_cpt_example(r) for r in cpt_val],   E.PREPARED_DIR/'cpt_val.jsonl')

In [ ]:
print("\n=== Creating Multi-task LoRA dataset ===")
multi_logs = [E.format_sft_example(r, 'logs') for r in logs_train]
multi_qa = [E.format_sft_example(r, 'qa') for r in qa_train]
multi = multi_logs + multi_qa
random.Random(PREP_SEED + 400).shuffle(multi)
E.write_jsonl(multi, E.PREPARED_DIR/'multitask_train_sft.jsonl')

print(f"Multi-task train: {len(multi)} (logs: {len(multi_logs)}, QA: {len(multi_qa)})")

In [ ]:
manifest = {
    'seed': PREP_SEED,
    'val_ratio': VAL_RATIO,
    'source': 'ArXiv QA Dataset (Navyasri12355/arxiv-qa-dataset)',
    'logs_train_total': len(logs_train),
    'logs_val_total': len(logs_val),
    'logs_test_total': len(logs_test),
    'qa_train_total': len(qa_train),
    'qa_val_total': len(qa_val),
    'qa_test_total': len(qa_test),
    'cpt_train_total': len(cpt_train),
    'cpt_val_total': len(cpt_val),
    'multitask_train_total': len(multi),
    'files': {p.name: str(p) for p in sorted(E.PREPARED_DIR.glob('*.jsonl'))}
}
(E.PREPARED_DIR/'data_manifest.json').write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), 
    encoding='utf-8'
)

In [ ]:
print("\n=== Statistics ===")
print(f"Logs train: {len(logs_train)}")
print(f"Logs val: {len(logs_val)}")
print(f"Logs test: {len(logs_test)}")
print(f"QA train: {len(qa_train)}")
print(f"QA val: {len(qa_val)}")
print(f"QA test: {len(qa_test)}")
print(f"CPT train: {len(cpt_train)}")
print(f"CPT val: {len(cpt_val)}")
print(f"Multi-task train: {len(multi)}")

print("\nLogs train distribution:")
print(pd.Series([r['output'] for r in logs_train]).value_counts())
print("\nLogs val distribution:")
print(pd.Series([r['output'] for r in logs_val]).value_counts())
print("\nLogs test distribution:")
print(pd.Series([r['output'] for r in logs_test]).value_counts())

print(f"\n✅ Data preparation complete!")
print(f"Files saved to: {E.PREPARED_DIR}")